# 00d - CDSE Sentinel-2 OData Downloader & Kaggle Packager

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook authenticates against Copernicus Data Space Ecosystem (CDSE) Keycloak OAuth2 API (`assemsaid35@gmail.com`), queries OData V4 catalog over Liwa Oasis for Sentinel-2 L2A scenes, streams full `.zip` SAFE archives via CDSE Zipper endpoint into `/kaggle/working/`, and packages it as an isolated Kaggle Dataset (`assemelqersh/cdse-sentinel2-liwa`).

The mechanistic justification for this step is: Direct Bearer authenticated streaming from `zipper.dataspace.copernicus.eu` downloads full Sentinel-2 L2A SAFE zip archives directly into Kaggle dataset outputs.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (requests, tqdm, pystac-client) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("requests", "requests")
install_if_missing("tqdm", "tqdm")

# Step 1: Environment Setup & CDSE Token Retrieval
# The mechanistic justification for this step is: Authenticating user credentials against identity.dataspace.copernicus.eu generates a valid Bearer access token for OData catalog queries and zipper streams.
import os
import sys
import json
import requests
from tqdm import tqdm

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "../data/processed/cdse-sentinel2-liwa"
os.makedirs(OUT_DIR, exist_ok=True)

CDSE_USERNAME = "assemsaid35@gmail.com"
CDSE_PASSWORD = "mL&3Wq!7SH26#Ac"

def get_cdse_token(username, password):
    token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    payload = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password"
    }
    try:
        res = requests.post(token_url, data=payload, timeout=15)
        if res.status_code == 200:
            print(f"[SUCCESS] Authenticated CDSE user '{username}'.")
            return res.json().get("access_token")
    except Exception as e:
        print(f"Token error: {e}")
    return None

token = get_cdse_token(CDSE_USERNAME, CDSE_PASSWORD)


In [ ]:
# Step 2: Query CDSE OData Catalogue & Download Sentinel-2 L2A Product Archives
# The mechanistic justification for this step is: Streaming full product zip archives from zipper.dataspace.copernicus.eu populates /kaggle/working/ with Sentinel-2 SAFE archives.
if token:
    odata_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    polygon_wkt = "SRID=4326;POLYGON((53.5 22.8, 54.2 22.8, 54.2 23.3, 53.5 23.3, 53.5 22.8))"
    filter_query = (
        "Collection/Name eq 'SENTINEL-2' and "
        "ContentDate/Start ge 2024-10-01T00:00:00.000Z and "
        f"OData.CSC.Intersects(area=geography'{polygon_wkt}') and "
        "Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/Value le 10.0)"
    )
    
    params = {
        "$filter": filter_query,
        "$top": 2,
        "$orderby": "ContentDate/Start desc"
    }
    headers = {"Authorization": f"Bearer {token}"}
    print("Searching CDSE OData Catalogue for Sentinel-2 L2A scenes...")
    odata_res = requests.get(odata_url, params=params, headers=headers)
    if odata_res.status_code == 200:
        products = odata_res.json().get("value", [])
        print(f"Found {len(products)} Sentinel-2 L2A scene(s) in CDSE.")
        for p in products:
            p_id = p.get('Id')
            p_name = p.get('Name')
            dl_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({p_id})/$value"
            save_path = os.path.join(OUT_DIR, f"{p_name}.zip")
            if not os.path.exists(save_path):
                print(f"Downloading CDSE SAFE Archive: {p_name}...")
                with requests.get(dl_url, headers=headers, stream=True) as stream_res:
                    if stream_res.status_code == 200:
                        total = int(stream_res.headers.get('content-length', 0))
                        with open(save_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
                            for chunk in stream_res.iter_content(chunk_size=1024 * 1024):
                                if chunk:
                                    f.write(chunk)
                                    bar.update(len(chunk))
                        mb = os.path.getsize(save_path) / (1024 * 1024)
                        print(f"Saved {save_path} ({mb:.2f} MB)")

cdse_meta = {
    "title": "CDSE Sentinel-2 L2A Liwa Oasis Dataset",
    "id": "assemelqersh/cdse-sentinel2-liwa",
    "licenses": [{"name": "CC-BY-4.0"}]
}
with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(cdse_meta, f, indent=2)
print(f"[SUCCESS] CDSE Dataset packaged cleanly in {OUT_DIR}")
